In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage  # noqa: F401
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from rich.console import Console
from rich.markdown import Markdown

# console = Console(force_jupyter=False)
console = Console()
load_dotenv(override=True)

True

### Model Call


In [2]:
llm = ChatOpenAI(model="gpt-5.4-nano")

message = "In 1 sentence what does mean for an AI agent to be autonmous"

reply = llm.invoke(message)

console.print(Markdown(reply.content))

For an AI agent to be autonomous means it can independently perceive its environment, make decisions, and take     
actions toward its goals without needing constant human guidance.

### Streaming


In [3]:
message = "Tell me a two line poem on autonomus agents"
for chunk in llm.stream(message):
    # console.print(chunk.content, end="", markup=False, highlight=False)
    print(chunk.content, end="", flush=True)

Autonomous agents roam, learning paths from every trail,  
Decisions made in silence—guiding futures without fail.

### Use any OpenAI compatible LLM


In [4]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
reply = openrouter_llm.invoke("In one sentence what is Claude?")
console.print(reply.content)

Claude is an AI assistant made by Anthropic that can help with a wide variety of tasks like writing, analysis, 
math, coding, and creative projects.

### Messages


In [5]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words"),
    HumanMessage("In which year is the Replubic of France founded"),
]

console.print(llm.invoke(messages).content)

1792, during the French Revolution.

In [6]:
messages_dict = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words"},
    {"role": "user", "content": "In which year is the Replubic of France founded"},
]

console.print(llm.invoke(messages_dict).content)

The Republic of France founded: 1792.

### Tools


In [7]:
@tool
def get_share_price(symbol: str) -> float:
    """Returns the current ticker price for a stock symbol"""
    prices = {"AAPL": 202.95, "GOOGL": 342.65, "AMZN": 132.68}
    return prices.get(symbol.upper(), 0.0)

In [8]:
console.print(get_share_price, style="white", highlight=False, markup=True)

StructuredTool(
    name='get_share_price',
    description='Returns the current ticker price for a stock symbol',
    args_schema=<class 'langchain_core.utils.pydantic.get_share_price'>,
    func=<function get_share_price at 0x113863270>
)

In [9]:
get_share_price.invoke("AAPL")

202.95

### LLM with tools


In [10]:
llm_with_tools = llm.bind_tools([get_share_price])

message = "What is the share price of Amazon?"

print(llm_with_tools.invoke(message).content)

In [11]:
llm_with_tools.invoke(message).tool_calls

[{'name': 'get_share_price',
  'args': {'symbol': 'AMZN'},
  'id': 'call_uhzHd7MDyRV01ageXQwnLUsF',
  'type': 'tool_call'}]

In [12]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage("What is the share price of Amazon?")]

ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

tools = {
    "get_share_price": get_share_price,
}

for tool_call in ai_message.tool_calls:
    tool = tools.get(tool_call["name"])

    if tool is not None:
        result = tool.invoke(tool_call["args"])
        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
            )
        )

final_response = llm_with_tools.invoke(messages)
console.print(Markdown(final_response.content))

Amazon (AMZN) is currently $132.68 per share.

### Structured Output


In [15]:
class Company(BaseModel):
    name: str
    ticker: str
    founded_year: int


structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon")

console.print(company, style="white", highlight=False)

Company(name='Amazon.com, Inc.', ticker='AMZN', founded_year=1994)